In [1]:
from __future__ import annotations
 
import numpy as np
import pandas as pd
from numpy.random import Generator, RandomState
 
# --- arch imports ---
from arch.univariate import ZeroMean, ConstantMean, ARX, GARCH
from arch.univariate.distribution import (
    Distribution,
    Normal,
    StudentsT,
    SkewStudent,
    GeneralizedError,
)

In [2]:
n=1000
mu=0.15
omega = 0.05
alpha = 0.05
beta = 0.90
distribution = None
seed=42

In [3]:
# --- Mean ---
m = ConstantMean()
mean_params = [mu]

# --- Volatility ---
m.volatility = GARCH(p=1, q=1)
vol_params = [omega, alpha, beta]

# --- Distribution ---
if distribution is None:
    distribution = Normal(seed=seed)
m.distribution = distribution
# --- Flat parameter vector: [mean_params..., vol_params..., dist_params...] ---
dist_params = list(m.distribution.starting_values(np.array([])))

params = mean_params + vol_params + dist_params

model = m
result = model.simulate(
        params,
        nobs=n,
        burn=100
    )

In [4]:
result

,data,volatility,errors
0,0.465674,0.935123,0.315674
1,1.441507,0.917601,1.291507
2,0.235515,0.944029,0.085515
3,0.744533,0.923276,0.594533
4,-1.723265,0.913711,-1.873265
...,...,...,...
995,2.127710,1.008786,1.977710
996,-1.173424,1.077707,-1.323424
997,-0.857700,1.087603,-1.007700
998,1.753000,1.079520,1.603000


In [1]:
from core.dgp import GARCHProcess
import numpy as np
from scipy import stats

In [2]:
dgp = GARCHProcess(mu=0.05, omega=0.05, alpha=0.10, beta=0.85,
             dist='normal').calibrate_params(0.15, 0.3)

In [11]:
rng = np.random.default_rng(42)
res = dgp.simulate(1000, rng)

In [12]:
res.mean(),res.std()

(np.float64(0.1440580765323071), np.float64(0.2908905778707369))

In [13]:
from arch import arch_model

am = arch_model(res, mean='Constant', vol='GARCH',p=1,q=1, dist='normal', rescale=True)

In [17]:
res_fit = am.fit(update_freq=0,disp=False)

In [21]:
res_fit.params

mu          1.458920
omega       1.422387
alpha[1]    0.139633
beta[1]     0.694718
Name: params, dtype: float64

In [22]:
res_fit.params.tolist()

[1.458919806004956,
 1.4223865299041283,
 0.13963322357956667,
 0.6947177032745095]

In [23]:
res_fit.params['alpha[1]']

np.float64(0.13963322357956667)

In [24]:
from core.models import GARCH11Model

model = GARCH11Model()

In [25]:
model.fit(res)

{'omega': np.float64(1.4223865299041283),
 'alpha': np.float64(0.13963322357956667),
 'beta': np.float64(0.6947177032745095),
 'skew': 0.06744140746491294,
 'exc_kurt': 0.44746189485031795}

In [ ]:
model.avar()